# DMEPOS by Referring Provider (No Service) labeling

This script uses the `dmepos_rfrr.csv` and `leie_with_valid_npi_clean.csv` and returns `dmepos_rfrr_labeled.csv`, the labeled by referring provider dataset.

In [1]:
import pandas as pd

df = pd.read_csv('/dsa/groups/casestudycf25/team02/silver/dmepos_rfrr.csv',dtype={'rfrg_rrvdr_state_fips':str,'rfrg_prvdr_zip5':str}) # ensure Rfrg_Prvdr_State_FIPS & Rfrg_Prvdr_Zip5 are imported as str
df.head()

/tmp/ipykernel_432/3273820603.py:3: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/dsa/groups/casestudycf25/team02/silver/dmepos_rfrr.csv',dtype={'rfrg_rrvdr_state_fips':str,'rfrg_prvdr_zip5':str}) # ensure Rfrg_Prvdr_State_FIPS & Rfrg_Prvdr_Zip5 are imported as str


,rfrg_npi,rfrg_prvdr_last_name_org,rfrg_prvdr_first_name,rfrg_prvdr_mi,rfrg_prvdr_crdntls,rfrg_prvdr_ent_cd,rfrg_prvdr_st1,rfrg_prvdr_st2,rfrg_prvdr_city,rfrg_prvdr_state_abrvtn,...,bene_cc_ph_hf_non_ihd_v2_pct,bene_cc_ph_hyperlipidemia_v2_pct,bene_cc_ph_hypertension_v2_pct,bene_cc_ph_ischemic_heart_v2_pct,bene_cc_ph_osteoporosis_v2_pct,bene_cc_ph_parkinson_v2_pct,bene_cc_ph_arthritis_v2_pct,bene_cc_ph_stroke_tia_v2_pct,bene_avg_risk_scre,year
0,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,-1.0,-1.000000,1.000000,-1.0,-1.0,0.0,-1.0,-1.0,1.942167,2021
1,1003000480,Rothchild,Kevin,B,md,I,12605 E 16th Ave,NaN,Aurora,CO,...,-1.0,-1.000000,-1.000000,-1.0,-1.0,-1.0,-1.0,-1.0,2.278200,2021
2,1003000522,Weigand,Frederick,J,md,I,1565 Saxon Blvd,Suite 102,Deltona,FL,...,-1.0,1.000000,0.846154,-1.0,-1.0,0.0,-1.0,-1.0,1.872308,2021
3,1003000530,Semonche,Amanda,M,do,I,1021 Park Ave,Suite 203,Quakertown,PA,...,-1.0,0.833333,0.777778,-1.0,-1.0,-1.0,-1.0,-1.0,2.144684,2021
4,1003000597,Kim,Dae,Y,mdphd,I,1145 S Utica Ave,Suite 202,Tulsa,OK,...,-1.0,-1.000000,-1.000000,-1.0,-1.0,-1.0,-1.0,-1.0,2.006500,2021


In [2]:
labs = pd.read_csv('/dsa/groups/casestudycf25/team02/silver/leie_with_valid_npi_clean.csv',dtype={'npi':int}) # make sure npi numbers are read as int
labs['excldate'] = labs.excldate.apply(lambda x: int(x[:4])) # this is different in Hellbender because the dates are formatted "m/d/yyyy" there
labs = labs[['npi','fraud_flag','excldate']]
labs.head()

,npi,fraud_flag,excldate
0,1861529091,1,2024
1,1366544512,1,2021
2,1306284369,1,2025
3,1376214585,1,2025
4,1326353459,1,2022


In [3]:
# merge labs w/ df on npi, retain all NPIs in df
df = pd.merge(labs, df, how='right', left_on='npi', right_on='rfrg_npi')
df.head()

,npi,fraud_flag,excldate,rfrg_npi,rfrg_prvdr_last_name_org,rfrg_prvdr_first_name,rfrg_prvdr_mi,rfrg_prvdr_crdntls,rfrg_prvdr_ent_cd,rfrg_prvdr_st1,...,bene_cc_ph_hf_non_ihd_v2_pct,bene_cc_ph_hyperlipidemia_v2_pct,bene_cc_ph_hypertension_v2_pct,bene_cc_ph_ischemic_heart_v2_pct,bene_cc_ph_osteoporosis_v2_pct,bene_cc_ph_parkinson_v2_pct,bene_cc_ph_arthritis_v2_pct,bene_cc_ph_stroke_tia_v2_pct,bene_avg_risk_scre,year
0,NaN,NaN,NaN,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,...,-1.0,-1.000000,1.000000,-1.0,-1.0,0.0,-1.0,-1.0,1.942167,2021
1,NaN,NaN,NaN,1003000480,Rothchild,Kevin,B,md,I,12605 E 16th Ave,...,-1.0,-1.000000,-1.000000,-1.0,-1.0,-1.0,-1.0,-1.0,2.278200,2021
2,NaN,NaN,NaN,1003000522,Weigand,Frederick,J,md,I,1565 Saxon Blvd,...,-1.0,1.000000,0.846154,-1.0,-1.0,0.0,-1.0,-1.0,1.872308,2021
3,NaN,NaN,NaN,1003000530,Semonche,Amanda,M,do,I,1021 Park Ave,...,-1.0,0.833333,0.777778,-1.0,-1.0,-1.0,-1.0,-1.0,2.144684,2021
4,NaN,NaN,NaN,1003000597,Kim,Dae,Y,mdphd,I,1145 S Utica Ave,...,-1.0,-1.000000,-1.000000,-1.0,-1.0,-1.0,-1.0,-1.0,2.006500,2021


In [4]:
# impute missing values
df['npi'] = df['npi'].fillna(df['rfrg_npi'].astype(int))
df['fraud_flag'] = df['fraud_flag'].fillna(0)
df['excldate'] = df['excldate'].fillna(0)
# change npi to integer type
df['npi'] = df.npi.astype(int)

df.head()

,npi,fraud_flag,excldate,rfrg_npi,rfrg_prvdr_last_name_org,rfrg_prvdr_first_name,rfrg_prvdr_mi,rfrg_prvdr_crdntls,rfrg_prvdr_ent_cd,rfrg_prvdr_st1,...,bene_cc_ph_hf_non_ihd_v2_pct,bene_cc_ph_hyperlipidemia_v2_pct,bene_cc_ph_hypertension_v2_pct,bene_cc_ph_ischemic_heart_v2_pct,bene_cc_ph_osteoporosis_v2_pct,bene_cc_ph_parkinson_v2_pct,bene_cc_ph_arthritis_v2_pct,bene_cc_ph_stroke_tia_v2_pct,bene_avg_risk_scre,year
0,1003000126,0.0,0.0,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,...,-1.0,-1.000000,1.000000,-1.0,-1.0,0.0,-1.0,-1.0,1.942167,2021
1,1003000480,0.0,0.0,1003000480,Rothchild,Kevin,B,md,I,12605 E 16th Ave,...,-1.0,-1.000000,-1.000000,-1.0,-1.0,-1.0,-1.0,-1.0,2.278200,2021
2,1003000522,0.0,0.0,1003000522,Weigand,Frederick,J,md,I,1565 Saxon Blvd,...,-1.0,1.000000,0.846154,-1.0,-1.0,0.0,-1.0,-1.0,1.872308,2021
3,1003000530,0.0,0.0,1003000530,Semonche,Amanda,M,do,I,1021 Park Ave,...,-1.0,0.833333,0.777778,-1.0,-1.0,-1.0,-1.0,-1.0,2.144684,2021
4,1003000597,0.0,0.0,1003000597,Kim,Dae,Y,mdphd,I,1145 S Utica Ave,...,-1.0,-1.000000,-1.000000,-1.0,-1.0,-1.0,-1.0,-1.0,2.006500,2021


In [5]:
def make_labels(fraud_flag, start_exc, data_yr):
    if start_exc >= data_yr:
        return fraud_flag
    else:
        return 0

df['target'] = df[['fraud_flag','excldate','year']].apply(lambda x: make_labels(*x), axis=1)
df.head()

,npi,fraud_flag,excldate,rfrg_npi,rfrg_prvdr_last_name_org,rfrg_prvdr_first_name,rfrg_prvdr_mi,rfrg_prvdr_crdntls,rfrg_prvdr_ent_cd,rfrg_prvdr_st1,...,bene_cc_ph_hyperlipidemia_v2_pct,bene_cc_ph_hypertension_v2_pct,bene_cc_ph_ischemic_heart_v2_pct,bene_cc_ph_osteoporosis_v2_pct,bene_cc_ph_parkinson_v2_pct,bene_cc_ph_arthritis_v2_pct,bene_cc_ph_stroke_tia_v2_pct,bene_avg_risk_scre,year,target
0,1003000126,0.0,0.0,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,...,-1.000000,1.000000,-1.0,-1.0,0.0,-1.0,-1.0,1.942167,2021,0.0
1,1003000480,0.0,0.0,1003000480,Rothchild,Kevin,B,md,I,12605 E 16th Ave,...,-1.000000,-1.000000,-1.0,-1.0,-1.0,-1.0,-1.0,2.278200,2021,0.0
2,1003000522,0.0,0.0,1003000522,Weigand,Frederick,J,md,I,1565 Saxon Blvd,...,1.000000,0.846154,-1.0,-1.0,0.0,-1.0,-1.0,1.872308,2021,0.0
3,1003000530,0.0,0.0,1003000530,Semonche,Amanda,M,do,I,1021 Park Ave,...,0.833333,0.777778,-1.0,-1.0,-1.0,-1.0,-1.0,2.144684,2021,0.0
4,1003000597,0.0,0.0,1003000597,Kim,Dae,Y,mdphd,I,1145 S Utica Ave,...,-1.000000,-1.000000,-1.0,-1.0,-1.0,-1.0,-1.0,2.006500,2021,0.0


In [6]:
# convert target to int and drop unnecessary cols
df = df.drop(['fraud_flag','excldate','rfrg_npi'], axis=1)
df['target'] = df.target.astype(int)

# save as csv
df.to_csv('/dsa/groups/casestudycf25/team02/silver/dmepos_rfrr_labeled.csv', index=False)

df.head()

,npi,rfrg_prvdr_last_name_org,rfrg_prvdr_first_name,rfrg_prvdr_mi,rfrg_prvdr_crdntls,rfrg_prvdr_ent_cd,rfrg_prvdr_st1,rfrg_prvdr_st2,rfrg_prvdr_city,rfrg_prvdr_state_abrvtn,...,bene_cc_ph_hyperlipidemia_v2_pct,bene_cc_ph_hypertension_v2_pct,bene_cc_ph_ischemic_heart_v2_pct,bene_cc_ph_osteoporosis_v2_pct,bene_cc_ph_parkinson_v2_pct,bene_cc_ph_arthritis_v2_pct,bene_cc_ph_stroke_tia_v2_pct,bene_avg_risk_scre,year,target
0,1003000126,Enkeshafi,Ardalan,NaN,md,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,-1.000000,1.000000,-1.0,-1.0,0.0,-1.0,-1.0,1.942167,2021,0
1,1003000480,Rothchild,Kevin,B,md,I,12605 E 16th Ave,NaN,Aurora,CO,...,-1.000000,-1.000000,-1.0,-1.0,-1.0,-1.0,-1.0,2.278200,2021,0
2,1003000522,Weigand,Frederick,J,md,I,1565 Saxon Blvd,Suite 102,Deltona,FL,...,1.000000,0.846154,-1.0,-1.0,0.0,-1.0,-1.0,1.872308,2021,0
3,1003000530,Semonche,Amanda,M,do,I,1021 Park Ave,Suite 203,Quakertown,PA,...,0.833333,0.777778,-1.0,-1.0,-1.0,-1.0,-1.0,2.144684,2021,0
4,1003000597,Kim,Dae,Y,mdphd,I,1145 S Utica Ave,Suite 202,Tulsa,OK,...,-1.000000,-1.000000,-1.0,-1.0,-1.0,-1.0,-1.0,2.006500,2021,0
